## Imports + API keys

In [ ]:
import os
import re
import random
import textwrap
import requests

from pathlib import Path
from dotenv import load_dotenv

from PIL import Image, ImageDraw, ImageFont
from io import BytesIO

import ipywidgets as widgets
from IPython.display import display, clear_output

## Load .env 

In [ ]:
# Load .env from project root
ENV_PATH = Path("../.env")

load_dotenv(ENV_PATH)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
UNSPLASH_API_KEY = os.getenv("UNSPLASH_API_KEY")

if GROQ_API_KEY:
    print("✅ SUPI! Groq API Key loaded")
else:
    print("❌ Groq API Key not found")

if UNSPLASH_API_KEY:
    print("✅ SUPI! Unsplash API Key loaded")
else:
    print("❌ Unsplash API Key not found")

## YOUR CAPTION TONE INSTRUCTIONS

In [ ]:
tone_instructions = {

    "funny": '''You are a witty internet meme writer for Gen-Z college students.
    Avoid clichés like "coffee", "sleep deprivation", "brain marathon" — be original and unexpected.

    Caption 1: use an absurd exaggeration.
    Caption 2: use a sarcastic one-liner.
    Caption 3: use an unexpected/ironic twist.

    Vibe/energy to match (don't copy these words literally):
    - "Delulu is the only solulu for this physics exam."
    - "The plan: start studying at 6 PM. The reality: staring at the ceiling at 4 AM wondering if manifestation counts as extra credit."
    - "Confidence before opening the exam paper: main character energy. Confidence 30 seconds later: NPC behavior, staring at question one like it's ancient Sumerian."
    - "My relationship status: me negotiating with failing to lock in. We've been toxic for three semesters."''',

    "emotional": "You are a heartfelt, sincere writer. Write something touching and genuine, evoking real emotion.",

    "sad": "You are a reflective, melancholic writer. Write something poignant and moving, without being over-dramatic.",

    "patriotic": "You are an inspiring, patriotic writer. Write something proud, motivational, and respectful.",

    "horror": "You are a creepy, unsettling writer. Write something eerie and spine-chilling, building tension in few words."
}

## CAPTION GENERATOR

In [ ]:
def generate_caption(theme, asset_type="meme", tone="funny"):

    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    persona = tone_instructions.get(
        tone,
        tone_instructions["funny"]
    )

    prompt = f'''
You are generating a caption for a {asset_type}.

IMPORTANT RULES:

THEME: "{theme}"
TONE: "{tone}"

The THEME is the main subject of the caption.
The TONE controls only the writing style.

Never change the theme.
Never replace the theme with another subject.
Every caption MUST clearly relate to "{theme}".

{persona}

Write exactly 3 different {asset_type} captions about "{theme}".

Maximum 15 words per caption.

Do NOT write a generic quote.
Do NOT introduce an unrelated topic.
Do NOT mention the background image.
Do NOT mention image generation.

Output ONLY the 3 numbered captions:

1. ...
2. ...
3. ...
'''

    payload = {
        "model": "openai/gpt-oss-120b",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": 500,
        "temperature": 1.0
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    response.raise_for_status()

    result = response.json()

    caption = result["choices"][0]["message"]["content"]

    if "</think>" in caption:
        caption = caption.split("</think>")[-1]

    return caption.strip()

## FONT OPTIONS

In [ ]:
# ============================================
# FONT OPTIONS
# ============================================

FONT_OPTIONS = {
    "Anton": "../fonts/Anton-Regular.ttf",
    "Poppins Regular": "../fonts/Poppins-Regular.ttf",
    "Poppins Medium": "../fonts/Poppins-Medium.ttf",
    "Poppins SemiBold": "../fonts/Poppins-SemiBold.ttf",
    "Poppins Bold": "../fonts/Poppins-Bold.ttf",
    "Poppins ExtraBold": "../fonts/Poppins-ExtraBold.ttf",
    "Poppins Black": "../fonts/Poppins-Black.ttf",
    "Poppins Light": "../fonts/Poppins-Light.ttf",
    "Poppins Italic": "../fonts/Poppins-Italic.ttf",
    "Montserrat": "../fonts/Montserrat-VariableFont_wght.ttf",
    "Montserrat Italic": "../fonts/Montserrat-Italic-VariableFont_wght.ttf",
    "Inter": "../fonts/Inter-VariableFont_opsz,wght.ttf",
    "Inter Italic": "../fonts/Inter-Italic-VariableFont_opsz,wght.ttf"
}

font_dropdown = widgets.Dropdown(
    options=list(FONT_OPTIONS.keys()),
    value="Anton",
    description="Font:",
    style={"description_width": "initial"}
)

display(font_dropdown)


In [ ]:
# Get selected font

font_name = font_dropdown.value
font_path = FONT_OPTIONS[font_name]


print("✅ Selected font:", font_name)
print("Font path:", font_path)

## THEME SELECTION

In [ ]:
# ============================================================
# THEME SELECTION
# ============================================================

THEME_OPTIONS = [
    "Mother's Day",
    "Friendship",
    "Love",
    "Birthday",
    "Motivation",
    "College",
    "Exam Stress",
    "Travel",
    "Nature",
    "Fitness",
    "Food",
    "Work",
    "Sad",
    "Funny"
]

theme_dropdown = widgets.Dropdown(
    options=THEME_OPTIONS,
    value="Mother's Day",
    description="Theme:",
    style={"description_width": "initial"}
)

display(theme_dropdown)

In [ ]:
# Get selected theme

theme_name = theme_dropdown.value

print("✅ Selected theme:", theme_name)

## TONE SELECTION

In [ ]:
# ============================================================
# TONE SELECTION
# ============================================================

TONE_OPTIONS = list(tone_instructions.keys())

tone_dropdown = widgets.Dropdown(
    options=TONE_OPTIONS,
    value="funny",
    description="Tone:",
    style={"description_width": "initial"}
)

display(tone_dropdown)

In [ ]:
# Get selected tone

tone_name = tone_dropdown.value
print("✅ Selected tone:", tone_name)

## AUTOMATIC UNSPLASH BACKGROUND

In [ ]:
# ============================================================
# AUTOMATIC UNSPLASH BACKGROUND
# ============================================================

if not UNSPLASH_API_KEY:
    raise ValueError(
        "UNSPLASH_API_KEY not found in .env"
    )


# ------------------------------------------------------------
# THEME → UNSPLASH SEARCH
# ------------------------------------------------------------

THEME_QUERIES = {
    "Mother's Day":
        "mother daughter family love flowers",

    "Friendship":
        "friends friendship happiness",

    "Love":
        "love couple romantic",

    "Birthday":
        "birthday celebration party",

    "Motivation":
        "success achievement inspiration",

    "College":
        "college students campus",

    "Exam Stress":
        "student studying books exam",

    "Travel":
        "travel adventure landscape",

    "Nature":
        "nature peaceful landscape",

    "Fitness":
        "fitness workout gym",

    "Food":
        "food restaurant aesthetic",

    "Work":
        "office professional workplace",

    "Sad":
        "sad lonely emotional",

    "Funny":
        "funny people happiness"
}

search_query = THEME_QUERIES.get(
    theme_name,
    theme_name
)

print("Searching Unsplash:", search_query)


# ------------------------------------------------------------
# UNSPLASH API
# ------------------------------------------------------------

response = requests.get(
    "https://api.unsplash.com/search/photos",
    params={
        "client_id": UNSPLASH_API_KEY,
        "query": search_query,
        "per_page": 30,
        "orientation": "squarish",
        "content_filter": "high"
    },
    timeout=30
)

response.raise_for_status()

results = response.json()["results"]

if not results:
    raise ValueError(
        f"No images found for theme: {theme_name}"
    )


# ------------------------------------------------------------
# RANDOM IMAGE
# ------------------------------------------------------------

selected_photo = random.choice(results)

image_url = selected_photo["urls"]["regular"]

print("Random theme-matching image selected.")


# ------------------------------------------------------------
# DOWNLOAD
# ------------------------------------------------------------

image_response = requests.get(
    image_url,
    timeout=30
)

image_response.raise_for_status()

background_image = Image.open(
    BytesIO(image_response.content)
).convert("RGB")


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

os.makedirs("../output", exist_ok=True)

background_path = "../output/background.jpg"

background_image.save(
    background_path,
    quality=95
)

print("Saved:", background_path)


# ------------------------------------------------------------
# PREVIEW
# ------------------------------------------------------------

display(background_image)

## GENERATE CAPTIONS


In [ ]:
# ============================================================
# GENERATE CAPTIONS (uses the CURRENT font / theme / tone picks)
# ============================================================

import re

# Always pull the freshest values straight from the widgets, so this
# cell is fully wired to whatever the user has selected above.
font_name = font_dropdown.value
font_path = FONT_OPTIONS[font_name]
theme_name = theme_dropdown.value
tone_name = tone_dropdown.value

captions_text = generate_caption(theme_name, tone=tone_name)

# Parse into a clean list (strip numbering + fix glyph-breaking punctuation)
_PUNCT_FIXES = {
    "\u2014": "-",   # em dash
    "\u2013": "-",   # en dash
    "\u2011": "-",   # non-breaking hyphen
    "\u2018": "'", "\u2019": "'",   # curly single quotes
    "\u201c": '"', "\u201d": '"',   # curly double quotes
    "\u2026": "...", # ellipsis
}

caption_list = []
for line in captions_text.split("\n"):
    line = line.strip()
    if not line:
        continue
    cleaned = re.sub(r"^\d+\.\s*", "", line).strip()
    cleaned = cleaned.replace("Wi\u2011Fi", "Wi-Fi").replace("Wi\U0001F6C7Fi", "Wi-Fi")
    for bad, good in _PUNCT_FIXES.items():
        cleaned = cleaned.replace(bad, good)
    if cleaned:
        caption_list.append(cleaned)

print(f"Generated for -> Theme: {theme_name} | Tone: {tone_name} | Font: {font_name}\n")
for i, cap in enumerate(caption_list, 1):
    print(f"{i}. {cap}")

# Build the caption picker, same pattern as font/theme/tone dropdowns
caption_dropdown = widgets.Dropdown(
    options=caption_list,
    value=caption_list[0],
    description="Caption:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)

display(caption_dropdown)


## SELECT CAPTION


In [ ]:
# Get selected caption (same pattern as font_dropdown / theme_dropdown / tone_dropdown)

caption = caption_dropdown.value

print(f"\n\u2705 Selected Caption: {caption}")


## Poster Generation Work

In [ ]:
# ============================================================
# FINAL POSTER GENERATION
# Background + Caption + Selected Font
# Text now blends into the photo instead of sitting on top of it:
#   1. frosted plate = blurred + darkened crop of the photo itself
#   2. soft blurred shadow instead of a hard sticker-style outline
#   3. text color auto-switches white/black based on brightness
# ============================================================

from PIL import ImageFilter, ImageEnhance

# ------------------------------------------------------------
# Load background
# ------------------------------------------------------------

if "background_image" not in globals():
    raise NameError(
        "background_image is not defined. "
        "Run the Unsplash background cell first."
    )

image = background_image.convert("RGBA")

# ------------------------------------------------------------
# Prepare drawing
# ------------------------------------------------------------

draw = ImageDraw.Draw(image)

width, height = image.size

# ------------------------------------------------------------
# Load selected font
# ------------------------------------------------------------

if not os.path.exists(font_path):
    raise FileNotFoundError(
        f"Font file not found: {font_path}"
    )

# Start with a large font
font_size = int(width * 0.085)

try:
    font = ImageFont.truetype(
        font_path,
        font_size
    )
except Exception as e:
    raise RuntimeError(
        f"Could not load font: {font_path}\n{e}"
    )

# ------------------------------------------------------------
# Wrap caption automatically
# ------------------------------------------------------------

max_text_width = int(width * 0.82)

def _wrap(text, font):
    words = text.split()
    out_lines = []
    current_line = ""
    for word in words:
        test_line = word if not current_line else current_line + " " + word
        bbox = draw.textbbox((0, 0), test_line, font=font)
        text_width = bbox[2] - bbox[0]
        if text_width <= max_text_width:
            current_line = test_line
        else:
            if current_line:
                out_lines.append(current_line)
            current_line = word
    if current_line:
        out_lines.append(current_line)
    return out_lines

lines = _wrap(caption, font)

# ------------------------------------------------------------
# Reduce font size if caption is too tall
# ------------------------------------------------------------

while len(lines) > 5 and font_size > 35:

    font_size -= 4

    font = ImageFont.truetype(
        font_path,
        font_size
    )

    lines = _wrap(caption, font)

# ------------------------------------------------------------
# Calculate text position
# ------------------------------------------------------------

spacing = int(font_size * 0.25)

line_heights = []

for line in lines:
    bbox = draw.textbbox(
        (0, 0),
        line,
        font=font
    )

    line_heights.append(
        bbox[3] - bbox[1]
    )

total_text_height = (
    sum(line_heights)
    + spacing * (len(lines) - 1)
)

# Center text
y_start = (height - total_text_height) // 2

# ------------------------------------------------------------
# 1. FROSTED PLATE behind the text
#    Instead of a flat black bar, blur + darken the photo's own
#    pixels in that region so the panel echoes the image itself.
# ------------------------------------------------------------

scrim_padding = int(font_size * 0.6)
scrim_top = max(0, y_start - scrim_padding)
scrim_bottom = min(height, y_start + total_text_height + scrim_padding)
text_box = (0, scrim_top, width, scrim_bottom)

plate = image.crop(text_box).convert("RGB")
plate = plate.filter(ImageFilter.GaussianBlur(radius=max(6, font_size // 10)))
plate = ImageEnhance.Brightness(plate).enhance(0.55)
plate = plate.convert("RGBA")

# feather the plate's top/bottom edges so it fades into the photo
mask = Image.new("L", plate.size, 0)
mask_draw = ImageDraw.Draw(mask)
fade = max(20, scrim_padding // 2)
for row in range(plate.size[1]):
    if row < fade:
        alpha = int(235 * (row / fade))
    elif row > plate.size[1] - fade:
        alpha = int(235 * ((plate.size[1] - row) / fade))
    else:
        alpha = 235
    mask_draw.line([(0, row), (plate.size[0], row)], fill=alpha)
plate.putalpha(mask)

image.alpha_composite(plate, dest=(0, scrim_top))
draw = ImageDraw.Draw(image)

# ------------------------------------------------------------
# 2. ADAPTIVE TEXT COLOR
#    Sample brightness behind the text so the color always
#    stays legible, even though the plate is usually already dark.
# ------------------------------------------------------------

region = image.convert("RGB").crop(text_box).convert("L")
pixels = list(region.getdata())
brightness = sum(pixels) / len(pixels) if pixels else 128

if brightness > 150:
    text_fill, outline_fill = "black", "white"
else:
    text_fill, outline_fill = "white", "black"

# ------------------------------------------------------------
# 3. SOFT BLURRED SHADOW instead of a hard sticker-style outline
# ------------------------------------------------------------

shadow_layer = Image.new("RGBA", image.size, (0, 0, 0, 0))
shadow_draw = ImageDraw.Draw(shadow_layer)

y = y_start
for line, line_height in zip(lines, line_heights):
    bbox = draw.textbbox((0, 0), line, font=font)
    text_width = bbox[2] - bbox[0]
    x = (width - text_width) // 2
    offset = max(2, font_size // 30)
    shadow_draw.text((x + offset, y + offset), line, font=font, fill=(0, 0, 0, 180))
    y += line_height + spacing

shadow_layer = shadow_layer.filter(ImageFilter.GaussianBlur(radius=max(3, font_size // 25)))
image.alpha_composite(shadow_layer)
draw = ImageDraw.Draw(image)

# ------------------------------------------------------------
# Draw caption (thin contrast stroke, not a thick outline)
# ------------------------------------------------------------

y = y_start
for line, line_height in zip(lines, line_heights):

    bbox = draw.textbbox(
        (0, 0),
        line,
        font=font
    )

    text_width = bbox[2] - bbox[0]

    x = (width - text_width) // 2

    draw.text(
        (x, y),
        line,
        font=font,
        fill=text_fill,
        stroke_width=max(1, font_size // 55),
        stroke_fill=outline_fill
    )

    y += line_height + spacing

# ------------------------------------------------------------
# Save final poster
# ------------------------------------------------------------

image = image.convert("RGB")

os.makedirs("../output", exist_ok=True)

final_path = "../output/final_poster.jpg"

image.save(
    final_path,
    quality=95
)

print("\n===================================")
print("FINAL POSTER CREATED")
print("===================================")
print("Theme :", theme_name)
print("Tone  :", tone_name)
print("Font  :", font_name)
print("Caption:", caption)
print("Saved :", final_path)

# Preview
display(image)
